In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset", split="train")

print(dataset)

In [ ]:
df = dataset.to_pandas()
print(f"Размер датасета: {df.shape}")
print("\nКолонки:", df.columns.tolist())
print("\nРаспределение классов:")
print(df['Email Type'].value_counts())

print("\nПример обычного письма:")
print(df[df['Email Type'] == 'Safe Email']['Email Text'].iloc[1][:500])

print("\nПример фишингового письма:")
print(df[df['Email Type'] == 'Phishing Email']['Email Text'].iloc[1][:500])

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, classification_report
)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['Email Text'].apply(clean_text)

df = df[df['clean_text'] != ""]

print(f"После очистки осталось {len(df)} писем")

In [ ]:
df = df.rename(columns={'Email Text': 'text', 'Email Type': 'label'})

df['label'] = df['label'].map({'Safe Email': 0, 'Phishing Email': 1})

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=3,
    max_df=0.9,
    stop_words='english',
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

print(f"Размер матрицы признаков: {X.shape}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} писем")
print(f"Test:  {X_test.shape[0]} писем")


In [ ]:
# Логистическая регрессия
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Дерево решений
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
print("МЕТРИКИ ОЦЕНКИ (для класса Phishing = 1)")

def evaluate_model(model, X_test, y_test, name):
    pred = model.predict(X_test)
    pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    print(f"\n{name}:")
    print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
    print(f"Precision: {precision_score(y_test, pred):.4f}")
    print(f"Recall (TPR): {recall_score(y_test, pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, pred):.4f}")

    # Специальные метрики темы
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    mcc = matthews_corrcoef(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba) if pred_proba is not None else None

    print(f"True Positive Rate (TPR / Recall):     {recall_score(y_test, pred):.4f}")
    print(f"False Negative Rate (FNR):             {fnr:.4f}")
    print(f"False Positive Rate (FPR):             {fpr:.4f}")
    print(f"Specificity:                           {specificity:.4f}")
    print(f"ROC-AUC:                               {roc_auc:.4f}" if roc_auc else "ROC-AUC: N/A")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")

# Оцениваем обе модели
evaluate_model(lr_model, X_test, y_test, "Logistic Regression")
evaluate_model(dt_model, X_test, y_test, "Decision Tree")

In [ ]:
print("МЕТРИКИ ОЦЕНКИ (для класса Safe Email = 0)")

def evaluate_model_safe(model, X_test, y_test, name):
    pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        pred_proba = model.predict_proba(X_test)[:, 0]   # вероятности класса 0
    else:
        pred_proba = None

    print(f"\n{name} (для Safe Email):")
    print(f"Accuracy:  {accuracy_score(y_test, pred):.4f}")
    print(f"Precision: {precision_score(y_test, pred, pos_label=0):.4f}")
    print(f"Recall (TPR): {recall_score(y_test, pred, pos_label=0):.4f}")
    print(f"F1-Score:  {f1_score(y_test, pred, pos_label=0):.4f}")

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    fpr = fn / (fn + tp) if (fn + tp) > 0 else 0
    specificity = tp / (tp + fn) if (tp + fn) > 0 else 0
    fnr = fp / (fp + tn) if (fp + tn) > 0 else 0
    mcc = matthews_corrcoef(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba) if pred_proba is not None else None

    print(f"True Positive Rate (TPR / Recall) для Safe:     {recall_score(y_test, pred, pos_label=0):.4f}")
    print(f"False Negative Rate (FNR) для Safe:             {fnr:.4f}")
    print(f"False Positive Rate (FPR) для Safe:             {fpr:.4f}")
    print(f"Specificity для Safe:                           {specificity:.4f}")
    print(f"ROC-AUC для Safe Email:                         {roc_auc:.4f}" if roc_auc else "ROC-AUC: N/A")
    print(f"Matthews Correlation Coefficient (MCC):         {mcc:.4f}")


# Оцениваем обе модели для класса Safe Email
evaluate_model_safe(lr_model, X_test, y_test, "Logistic Regression")
evaluate_model_safe(dt_model, X_test, y_test, "Decision Tree")

In [ ]:
print("ГРАФИК ROC-AUC ДЛЯ RL")

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Создаём график
plt.figure(figsize=(10, 8))

# Для LR
lr_proba = lr_model.predict_proba(X_test)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
roc_auc_lr = auc(fpr_lr, tpr_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', linewidth=2)

# Диагональ случайного классификатора
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)')

# Оформление графика
plt.title('ROC-AUC Curve for Phishing Detection', fontsize=14)
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR / Recall)', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Logistic Regression AUC: {roc_auc_lr:.4f}")

In [ ]:
print("CONFUSION MATRIX ДЛЯ ОБЕИХ МОДЕЛЕЙ")

import seaborn as sns

def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Safe Email', 'Phishing Email'],
                yticklabels=['Safe Email', 'Phishing Email'])

    plt.title(f'Confusion Matrix — {model_name}')
    plt.xlabel('Предсказано')
    plt.ylabel('Реально')
    plt.tight_layout()
    plt.show()

# Предсказания
lr_pred = lr_model.predict(X_test)
dt_pred = dt_model.predict(X_test)

# матрицы
plot_confusion_matrix(y_test, lr_pred, "Logistic Regression")
plot_confusion_matrix(y_test, dt_pred, "Decision Tree")

In [ ]:
print("ПРИМЕРЫ РАБОТЫ МОДЕЛИ на тестовых данных")

test_df = pd.DataFrame({
    'text': df.loc[y_test.index, 'text'].values,
    'clean_text': df.loc[y_test.index, 'clean_text'].values,
    'true_label': y_test.values
})

examples = test_df.sample(5, random_state=42)

for i, row in examples.iterrows():
    text = row['text'][:300] + "..." if len(row['text']) > 300 else row['text']
    true_label = "Phishing" if row['true_label'] == 1 else "Safe"

    vec = vectorizer.transform([row['clean_text']])
    pred = lr_model.predict(vec)[0]
    pred_label = "Phishing" if pred == 1 else "Safe"

    print(f"Пример {i+1}:")
    print(f"Текст: {text}")
    print(f"Реально:      {true_label}")
    print(f"Предсказано:  {pred_label}")
    print("-" * 80)